# CCTV Model Selection Overview

목표: 후보를 한 점수표로 섞지 않고 역할·평가 범위·승격 권한을 분리해 현재 결정을 재현한다.

## 실험 질문과 성공 기준

- 속성 proxy, strict ReID, 생성형 검토의 지표를 같은 의미로 해석하지 않는다.
- 실행 실패·미측정 후보는 0점으로 바꾸지 않는다.
- strict ReID가 자동 매칭 gate를 통과하지 못하면 Top-K 검색으로만 제한한다.
- promotion은 독립 identity label, heldout, 사람 검토와 provenance가 모두 있을 때만 가능하다.

In [1]:
from pathlib import Path
import json

repo_root = Path.cwd()
snapshot_path = repo_root / 'configs' / 'model_selection_snapshot.json'
if not snapshot_path.is_file():
    repo_root = repo_root.parent
    snapshot_path = repo_root / 'configs' / 'model_selection_snapshot.json'

snapshot = json.loads(snapshot_path.read_text(encoding='utf-8'))
{
    'roles': snapshot['roles'],
    'runtime': snapshot['runtimePolicy'],
    'automaticIdentityMatch': snapshot['strictReidDecision']['automaticIdentityMatch'],
    'promotion': snapshot['promotion']['status'],
}

{'roles': {'embeddedCandidate': {'candidate': 'student_CLIP_hard',
   'mode': 'first_pass_attribute_candidate',
   'status': 'proxy_winner_not_production_approved'},
  'serverAttribute': {'candidate': 'SOLIDER Swin-B + PAR',
   'mode': 'structured_multi_label_attribute',
   'status': 'selected_for_implementation_not_production_approved'},
  'reid': {'candidateFamily': 'SOLIDER or TransReID family',
   'observedStrictCandidate': 'SOLIDER-ReID Swin-B Top-3 mean',
   'mode': 'top_k_retrieval_only',
   'status': 'candidate_retriever'},
  'generativeReview': {'candidate': 'Qwen family',
   'mode': 'conflict_and_low_confidence_review',
   'status': 'review_only'}},
 'runtime': {'status': 'provisional',
  'runtimeEnforced': True,
  'productionApproved': False,
  'retrievalOnlyEnforced': True,
  'generativePrimaryClassifier': False,
  'automaticGenerativeFallback': False,
  'generativeRequiredForAutoMatch': False},
 'automaticIdentityMatch': 'BLOCKED',
 'promotion': 'NOT_APPROVED'}

## 현재 결정

속성 proxy의 CLIP 점수는 임베디드 후보 선택 근거로만 남긴다. strict ReID의 SOLIDER 결과는 Top-K 검색 후보로 유지하지만, 자동 동일인 매칭은 `BLOCKED`다. Qwen 계열은 충돌·저신뢰도 검토 보조이며 최종 판정 점수에 직접 합산하지 않는다.

In [2]:
attribute_results = [
    candidate
    for candidate in snapshot['historicalAttributeProxyComparison']['candidates']
    if candidate['status'] == 'measured'
]
{
    'currentAttributeProxy': snapshot['currentAttributeProxy'],
    'historicalAttributeProxyMeasured': attribute_results,
    'strictReid': snapshot['strictReidDecision'],
    'nextRequirements': snapshot['promotion']['requires'],
}

{'currentAttributeProxy': {'scope': 'PA-100K local 100-image subset with six fields and deterministic image-level folds',
  'metric': 'per_field_top1_mean',
  'bestObserved': 0.72666668,
  'notEquivalentTo': ['CCTV identity accuracy',
   'track-heldout identity result',
   'automatic match eligibility',
   'external mA or InsF1']},
 'historicalAttributeProxyMeasured': [{'candidate': 'CLIP ViT-L/14',
   'status': 'measured',
   'attributeScore': 0.414,
   'p95Seconds': 4.141},
  {'candidate': 'Qwen3-VL-2B',
   'status': 'measured',
   'attributeScore': 0.393,
   'p95Seconds': 8.298}],
 'strictReid': {'protocol': 'cross-camera and cross-sequence gallery exclusion',
  'candidate': 'SOLIDER-ReID Swin-B Top-3 mean',
  'rank1': 0.4737,
  'recallAt5': 0.7789,
  'identityMrr': 0.6074,
  'candidateRetrieverEnabled': True,
  'automaticIdentityMatch': 'BLOCKED',
  'reason': 'The strict result is below the automatic-match gate and must not be replaced with an overlapping-gallery result.'},
 'nextR

## 다음 루프

독립 교차 카메라·이벤트 identity label을 확보한 뒤, group·track·시간 누수를 막은 동일 프로토콜에서 후보 하나의 변경만 다시 측정한다. 그 결과가 artifact와 사람 검토로 재현될 때에만 promotion gate를 다시 평가한다.